In [1]:
import gurobipy as gp
import numpy as np
import pandas as pd
from gurobipy import GRB
import rasterio

In [2]:
tif_path = "ForUSTree_2018_HighVeg_TreeCoverage.tif"

with rasterio.open(tif_path) as src:
    A = src.read(1)          # first band as a 2D numpy array
    nodata = src.nodata
    transform = src.transform
    crs = src.crs

print("Shape (rows, cols):", A.shape)
print("dtype:", A.dtype)
print("nodata:", nodata)
print("CRS:", crs)

n, p = A.shape

Shape (rows, cols): (1376, 1542)
dtype: uint8
nodata: 0.0
CRS: EPSG:6344


In [8]:
#reading data
src = rasterio.open("ForUSTree_2018_HighVeg_TreeCoverage.tif")
b = src.read(1)

src = rasterio.open("texas_clipped.tif")
imp = src.read(1)


In [9]:
#Placeholder for material cost of planting 1 tree
t = 10

#Placeholder for site cost of planting 1 tree

siteCost = 20

#Placeholder for max # of trees at 1 location
maxT = 100

#Placeholder for budget

budget = 1000000

HI1 = 0.5
HI2 = 2.5


#Data

#Placeholder for Impervious
a = np.random.randint(0, 2, size=(n, p))


In [5]:




# -----------------------------
# Parameters
# -----------------------------
t = 10
siteCost = 20
maxT = 100
budget = 1_000_000

gamma = 0.01   # canopy fraction gained per tree (1 tree = +1% canopy)

n = 1376
p = 1542

# -----------------------------
# Data placeholders
# Replace these with your real arrays
# -----------------------------
a_base = np.ones((n, p), dtype=int)    # base allowed-site mask (0/1)
#b = np.random.uniform(0.0, 0.60, size=(n, p))      # baseline canopy fraction in [0,1]
#imp = np.random.uniform(0.0, 1.0, size=(n, p))     # imperviousness in [0,1]


#Imperviousness must be less than 0.85 to plant
imp_threshold = 0.85
a = a_base * (imp <= imp_threshold).astype(int)

#Allowing the imperviousness to effect the cooling effect of trees
HI1_grid = np.zeros((n, p))
HI2_grid = np.zeros((n, p))

low_imp = imp < 0.25
mid_imp = (imp >= 0.25) & (imp < 0.50)
high_imp = imp >= 0.50

# Per +1% canopy added, expressed in degrees C
# Stored as "per fraction canopy", so divide by 100
HI1_grid[low_imp]  = 0.75 / 100
HI2_grid[low_imp]  = 2.00 / 100

HI1_grid[mid_imp]  = 0.30 / 100
HI2_grid[mid_imp]  = 2.50 / 100

HI1_grid[high_imp] = 0.10 / 100
HI2_grid[high_imp] = 1.80 / 100


beta = 1.0 + 0.75 * imp

# -----------------------------
# Band setup from baseline canopy b
# -----------------------------
u = np.maximum(0.40 - b, 0.0)     # room left before hitting 40% canopy
Mtot = np.maximum(0.80 - b, 0.0)  # total room left before hitting 80% canopy

# -----------------------------
# Model
# -----------------------------
model = gp.Model("MILP_imp_combined")

# -----------------------------
# Variables
# -----------------------------
x = model.addMVar(shape=(n, p), vtype=GRB.INTEGER, lb=0, name="x")
y = model.addMVar(shape=(n, p), vtype=GRB.BINARY, name="y")
addCanopy = model.addMVar(shape=(n, p), vtype=GRB.CONTINUOUS, lb=0.0, name="addCanopy")
z1 = model.addMVar(shape=(n, p), vtype=GRB.CONTINUOUS, lb=0.0, name="z_band1")
z2 = model.addMVar(shape=(n, p), vtype=GRB.CONTINUOUS, lb=0.0, name="z_band2")
w = model.addMVar(shape=(n, p), vtype=GRB.BINARY, name="cross40")

# -----------------------------
# Constraints
# -----------------------------

# Budget
model.addConstr(t * x.sum() + siteCost * y.sum() <= budget, name="budget")

# Activation: only plant at activated sites
model.addConstr(x <= maxT * y, name="maxTreesPerSite")

# Feasibility mask (Strategy 1 included here)
model.addConstr(y <= a, name="allowedSites")

# Trees -> added canopy
model.addConstr(addCanopy == gamma * x, name="canopy_from_trees")

# Cap total canopy at 80%
model.addConstr(addCanopy <= Mtot, name="cap_at_80")

# Split added canopy into the two canopy bands
model.addConstr(z1 + z2 == addCanopy, name="split_canopy")

# Band 1 can only fill remaining room up to 40%
model.addConstr(z1 <= u, name="band1_cap")

# Logic so band 2 is used only if band 1 is filled
model.addConstr(addCanopy <= u + Mtot * w, name="cross_logic_a")
model.addConstr(z2 <= Mtot * w, name="cross_logic_z2")
model.addConstr(z1 >= u * w, name="cross_logic_fill1")
model.addConstr(z1 <= addCanopy, name="z1_le_add")

# -----------------------------
# Objective
# -----------------------------
degrees_cooled = 100.0 * ((beta * HI1_grid * z1).sum() + (beta * HI2_grid * z2).sum())

model.setObjective(degrees_cooled, GRB.MAXIMIZE)

# -----------------------------
# Optimize
# -----------------------------
model.optimize()

# -----------------------------
# Example outputs
# -----------------------------
if model.Status == GRB.OPTIMAL:
    x_sol = x.X
    y_sol = y.X
    z1_sol = z1.X
    z2_sol = z2.X

    print(f"Optimal objective (weighted cooling): {model.ObjVal:.4f}")
    print(f"Total trees planted: {x_sol.sum():.0f}")
    print(f"Activated sites: {(y_sol > 0.5).sum()}")

    selected = y_sol > 0.5
    if selected.sum() > 0:
        print(f"Average imperviousness of selected sites: {imp[selected].mean():.4f}")
        print(f"Average baseline canopy of selected sites: {b[selected].mean():.4f}")
else:
    print(f"Model ended with status code {model.Status}")

Set parameter Username
Set parameter LicenseID to value 2688724
Academic license - for non-commercial use only - expires 2026-07-16
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[arm] - Darwin 24.6.0 24G517)

CPU model: Apple M4 Pro
Thread count: 14 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 21217921 rows, 12730752 columns and 40758330 nonzeros
Model fingerprint: 0xafcebf97
Variable types: 6365376 continuous, 6365376 integer (4243584 binary)
Coefficient statistics:
  Matrix range     [1e-02, 1e+02]
  Objective range  [1e-01, 3e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [4e-01, 1e+06]
Found heuristic solution: objective -0.0000000
Presolve removed 12366138 rows and 5354267 columns (presolve time = 5s)...
Presolve removed 13841435 rows and 6829564 columns (presolve time = 11s)...
Presolve removed 13841435 rows and 6829564 columns (presolve time = 15s)...
Presolve removed 13841435 rows and 6829564 columns (presolve time = 2

: 

In [ ]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB

# -----------------------------
# Parameters
# -----------------------------
siteCost = 20
maxT = 100
budget = 1_000_000

n = 1376
p = 1542

# -----------------------------
# Tree types
# -----------------------------
tree_types = ["3gal", "5gal", "10gal"]
K = len(tree_types)

# cost per tree type
t = np.array([8, 12, 20], dtype=float)

# canopy fraction gained per tree type
# example: 3gal adds 0.6%, 5gal adds 1.0%, 10gal adds 1.8%
gamma = np.array([0.006, 0.010, 0.018], dtype=float)

# -----------------------------
# Data placeholders
# Replace these with real arrays
# -----------------------------
a_base = np.ones((n, p), dtype=int)
# b = np.random.uniform(0.0, 0.60, size=(n, p))
# imp = np.random.uniform(0.0, 1.0, size=(n, p))

# Imperviousness threshold
imp_threshold = 0.85
a = a_base * (imp <= imp_threshold).astype(int)

# -----------------------------
# Imperviousness-adjusted cooling coefficients
# -----------------------------
HI1_grid = np.zeros((n, p))
HI2_grid = np.zeros((n, p))

low_imp = imp < 0.25
mid_imp = (imp >= 0.25) & (imp < 0.50)
high_imp = imp >= 0.50

HI1_grid[low_imp]  = 0.75 / 100
HI2_grid[low_imp]  = 2.00 / 100

HI1_grid[mid_imp]  = 0.30 / 100
HI2_grid[mid_imp]  = 2.50 / 100

HI1_grid[high_imp] = 0.10 / 100
HI2_grid[high_imp] = 1.80 / 100

beta = 1.0 + 0.75 * imp

# -----------------------------
# Band setup from baseline canopy b
# -----------------------------
u = np.maximum(0.40 - b, 0.0)
Mtot = np.maximum(0.80 - b, 0.0)

# -----------------------------
# Model
# -----------------------------
model = gp.Model("MILP_multi_tree_types")

# -----------------------------
# Variables
# -----------------------------
# x[i,j,k] = number of trees of type k planted at site (i,j)
x = model.addMVar(shape=(n, p, K), vtype=GRB.INTEGER, lb=0, name="x")

# y[i,j] = whether site (i,j) is activated
y = model.addMVar(shape=(n, p), vtype=GRB.BINARY, name="y")

# total added canopy at each site
addCanopy = model.addMVar(shape=(n, p), vtype=GRB.CONTINUOUS, lb=0.0, name="addCanopy")

# band split
z1 = model.addMVar(shape=(n, p), vtype=GRB.CONTINUOUS, lb=0.0, name="z_band1")
z2 = model.addMVar(shape=(n, p), vtype=GRB.CONTINUOUS, lb=0.0, name="z_band2")
w = model.addMVar(shape=(n, p), vtype=GRB.BINARY, name="cross40")

# -----------------------------
# Constraints
# -----------------------------

# Budget: tree purchase costs + fixed site activation costs
model.addConstr(
    gp.quicksum(t[k] * x[:, :, k].sum() for k in range(K)) + siteCost * y.sum() <= budget,
    name="budget"
)

# Activation: total number of trees at a site only if site is activated
model.addConstr(
    x.sum(axis=2) <= maxT * y,
    name="maxTreesPerSite"
)

# Feasibility mask
model.addConstr(y <= a, name="allowedSites")

# Trees -> added canopy
model.addConstr(
    addCanopy == gp.quicksum(gamma[k] * x[:, :, k] for k in range(K)),
    name="canopy_from_trees"
)

# Cap total canopy at 80%
model.addConstr(addCanopy <= Mtot, name="cap_at_80")

# Split added canopy into two canopy bands
model.addConstr(z1 + z2 == addCanopy, name="split_canopy")

# Band 1 fills up to 40%
model.addConstr(z1 <= u, name="band1_cap")

# Logic for entering band 2 only after filling band 1
model.addConstr(addCanopy <= u + Mtot * w, name="cross_logic_a")
model.addConstr(z2 <= Mtot * w, name="cross_logic_z2")
model.addConstr(z1 >= u * w, name="cross_logic_fill1")
model.addConstr(z1 <= addCanopy, name="z1_le_add")

# -----------------------------
# Objective
# -----------------------------
degrees_cooled = 100.0 * ((beta * HI1_grid * z1).sum() + (beta * HI2_grid * z2).sum())
model.setObjective(degrees_cooled, GRB.MAXIMIZE)

# -----------------------------
# Optimize
# -----------------------------
model.optimize()

# -----------------------------
# Example outputs
# -----------------------------
if model.Status == GRB.OPTIMAL:
    x_sol = x.X
    y_sol = y.X
    z1_sol = z1.X
    z2_sol = z2.X

    print(f"Optimal objective (weighted cooling): {model.ObjVal:.4f}")
    print(f"Total trees planted: {x_sol.sum():.0f}")
    print(f"Activated sites: {(y_sol > 0.5).sum()}")

    for k, name in enumerate(tree_types):
        print(f"Total {name} trees planted: {x_sol[:, :, k].sum():.0f}")

    selected = y_sol > 0.5
    if selected.sum() > 0:
        print(f"Average imperviousness of selected sites: {imp[selected].mean():.4f}")
        print(f"Average baseline canopy of selected sites: {b[selected].mean():.4f}")
else:
    print(f"Model ended with status code {model.Status}")

Set parameter Username
Set parameter LicenseID to value 2688724
Academic license - for non-commercial use only - expires 2026-07-16


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x10569ba10>>
Traceback (most recent call last):
  File "/Users/devin/Library/Python/3.11/lib/python/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 
